In [24]:
import numpy as np
import torch
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [48]:
class NeuralNetwork:
    def __init__(self, layers):
        self.layers = len(layers) - 1
        self.weights = [
            np.random.randn(layers[i], layers[i+1]) * 0.1 for i in range(self.layers)
        ]
        # Notice the double parentheses ((1, layers[i+1]))
        self.bias = [
            np.zeros((1, layers[i+1])) for i in range(self.layers)
        ]
        self.activations = None
        self.z_values = None
        self.loss_history = []
    
    def _relu(self, x):
        return np.maximum(0, x)
    def _relu_derivative(self, x):
        return np.where(x > 0, 1, 0)
    def _calculate_loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)
    
    def forward(self, X):
        self.activations = [X]
        self.z_values = []

        for i in range(self.layers):
            z = np.dot(self.activations[-1], self.weights[i]) + self.bias[i]
            self.z_values.append(z)
            if i < self.layers - 1:
                a = self._relu(z)
            else:
                a = z
            self.activations.append(a)
    
    def backward(self, y_true, learning_rate):
        m = y_true.shape[0]
        delta = (2/m) * (self.activations[-1] - y_true)
        for i in reversed(range(self.layers)):
            current_activation = self.activations[i]
            grad_w = np.dot(current_activation.T, delta)
            grad_b = np.sum(delta, axis=0, keepdims=True)
            if i > 0:
                z_prev = self.z_values[i-1]
                delta = (2/m) * np.dot(delta, self.weights[i].T) * self._relu_derivative(z_prev)
            self.weights[i] -= learning_rate * grad_w
            self.bias[i] -= learning_rate * grad_b
    
    def fit(self, X, y, epochs=1000, learning_rate=0.01):
        self.loss_history = []
        for epoch in range(epochs):
            self.forward(X)
            self.backward(y, learning_rate)
            loss = self._calculate_loss(y, self.activations[-1])
            self.loss_history.append(loss)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {loss}")

    def predict(self, X):
        self.forward(X)
        return self.activations[-1]


In [26]:
@dataclass
class ModelConfig:
    layers: list
    batch_size: int = 32
    learning_rate: float = 0.01
    epochs: int = 100
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
config = ModelConfig(layers=[2, 4, 1], learning_rate=0.01, epochs=100)

In [27]:
class NeuralNetworkv1:
    def __init__(self, layers, learning_rate=0.01):
        self.layers = len(layers) - 1
        self.weights = [
            (torch.randn(layers[i], layers[i+1], device=config.device) * 0.1).requires_grad_() for i in range(self.layers)
        ]
        self.biases = [
            (torch.randn(layers[i+1], device=config.device) * 0.1).requires_grad_() for i in range(self.layers)
        ]
        self.optimizer = torch.optim.SGD(self.weights + self.biases, lr=learning_rate)
        self.loss_history = []
        self.activations = None
    
    def _relu(self, x):
        return torch.relu(x)
    def _calculate_loss(self, y_true, y_pred):
        return torch.mean((y_true.view_as(y_pred) - y_pred) ** 2)
        return torch.mean((y_true - y_pred) ** 2)
    
    def forward(self, X):
        self.activations = [X]
        for i in range(self.layers):
            Z = torch.mm(self.activations[-1], self.weights[i]) + self.biases[i]
            if i < self.layers - 1:
                A = self._relu(Z)
            else:
                A = Z
            self.activations.append(A)
    
    # def fit(self, X, y, epochs=100):
    #     self.loss_history = []
    #     for epoch in range(epochs):
    #         self.optimizer.zero_grad()
    #         self.forward(X)
    #         loss = self._calculate_loss(y, self.activations[-1])
    #         loss.backward()
    #         self.optimizer.step()
    #         self.loss_history.append(loss)

    #         if (epoch + 1) % 10 == 0:
    #             print(f"Epoch {epoch + 1}, Loss: {loss.item()}")
    
    def fit(self, dataloader, epochs=100):
        self.loss_history = []
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            
            # Iterate through the mini-batches provided by the DataLoader
            for batch_X, batch_y in dataloader:
                self.optimizer.zero_grad()
                
                # Forward pass on just the mini-batch
                self.forward(batch_X)
                loss = self._calculate_loss(batch_y, self.activations[-1])
                
                # Backward pass and update weights
                loss.backward()
                self.optimizer.step()
                
                # Accumulate the loss to calculate the average later
                epoch_loss += loss.item()
                
            # Calculate the average loss across all batches for this epoch
            avg_loss = epoch_loss / len(dataloader)
            self.loss_history.append(avg_loss)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    def predict(self, X):
        with torch.no_grad():
            self.forward(X)
            return self.activations[-1].numpy()

In [28]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        """whatever you want to initialize and transform"""
        self.X = torch.tensor(X, dtype=torch.float32, device=config.device)
        self.y = torch.tensor(y, dtype=torch.float32, device=config.device)

    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [37]:
data = fetch_california_housing()
X, y = data.data, data.target
scaler = StandardScaler(copy=False)
X = scaler.fit_transform(X)
y = scaler.fit_transform(y.reshape(-1, 1))
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
dataset_ = CustomDataset(x_train, y_train)
dataLoader = DataLoader(dataset_, batch_size=config.batch_size, shuffle=True)

In [31]:
nn = NeuralNetworkv1(layers=[X.shape[1], 64, 32, 1], learning_rate=config.learning_rate)
nn.fit(dataLoader, epochs=config.epochs)

Epoch 10, Loss: 0.2708
Epoch 20, Loss: 0.2381
Epoch 30, Loss: 0.2233
Epoch 40, Loss: 0.2163
Epoch 50, Loss: 0.2084
Epoch 60, Loss: 0.2042
Epoch 70, Loss: 0.2019
Epoch 80, Loss: 0.1985
Epoch 90, Loss: 0.1966
Epoch 100, Loss: 0.1956


In [33]:
pred = nn.predict(torch.tensor(x_test, dtype=torch.float32, device=config.device))
mse = mean_squared_error(y_test, pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 0.20975305912557854


In [52]:
model = NeuralNetwork(layers=[X.shape[1], 64, 64, 1])
model.fit(x_train, y_train, epochs=100, learning_rate=0.5)

Epoch 10, Loss: 0.7442582539763254
Epoch 20, Loss: 0.6335243548374763
Epoch 30, Loss: 0.5836527992617196
Epoch 40, Loss: 0.5567000712985892
Epoch 50, Loss: 0.5392991895081449
Epoch 60, Loss: 0.5263545261713597
Epoch 70, Loss: 0.5158000935995801
Epoch 80, Loss: 0.506739034338898
Epoch 90, Loss: 0.49874437168448577
Epoch 100, Loss: 0.4915860826291576
